# 第5回 データベースと Python の連携

**データ分析（da）／ 情報工学部 情報工学科 2年次 前期 ／ 担当: 後藤 弘光**

---

これまでは CSV ファイルや Web API からデータを取り込んできました。しかし実務で扱うデータの多くは **データベース（DB）**に蓄積されています。今回は Python から直接 DB に接続し、SQL クエリで必要な部分だけを抽出して DataFrame として扱う方法を学びます。

## 1. 要点まとめ

### 1.1 データベースとは

**データベース (Database, DB)** は、大量のデータを効率的に保存・検索・更新するための仕組みです。特に **リレーショナル DB (RDB)** では、データが「表（テーブル）」として保存されており、**SQL (Structured Query Language)** という問い合わせ言語で必要な行・列だけを取り出せます。

本講義では Python の標準ライブラリに同梱されている **SQLite** を使います。SQLite は 1 ファイル完結型の軽量 DB で、追加インストール不要で手軽に試せます。

### 1.2 Python で使うライブラリ

| ライブラリ | 役割 |
| :-- | :-- |
| `sqlite3` | Python 標準の SQLite クライアント。接続 (`connect`) と切断 (`close`) を担当 |
| `pandas.read_sql_query()` | SQL 文字列と接続オブジェクトを渡すと、結果を DataFrame として返す |

基本の流れは **接続 → SQL 実行 → 切断** の 3 ステップです。下の準備セルで接続だけ作っておき、次節からは異なる SQL を順に試していきます。

In [ ]:
# 準備：ライブラリ読み込みと DB 接続
import sqlite3
import pandas as pd

conn = sqlite3.connect('data/sample_sales.db')

### 1.3 SQL を段階的に学ぶ

配布データベース `sample_sales.db` には `sales` というテーブルが 1 つ入っています（`date, product, category, price, quantity` の 5 列）。4 つの SQL を順に試しながら、少しずつ書き方を覚えていきましょう。

#### ステップ 1：全列を取り出す（SELECT * FROM）

最もシンプルな SQL です。`SELECT *` で全列、`FROM sales` で対象テーブルを指定し、`LIMIT 5` で先頭 5 行だけに絞ります（データ量が多いのでまずは覗き見から始める習慣をつけましょう）。

In [ ]:
pd.read_sql_query('SELECT * FROM sales LIMIT 5', conn)

#### ステップ 2：必要な列だけ取り出す（列名の指定）

`SELECT` の後ろに列名をカンマ区切りで並べると、その列だけが返ってきます。

In [ ]:
pd.read_sql_query('SELECT product, price, quantity FROM sales LIMIT 5', conn)

#### ステップ 3：条件で絞り込む（WHERE）

`WHERE` の後ろに条件式を書くと、条件を満たす行だけが返ります。文字列の値はシングルクォートで囲みます。

In [ ]:
pd.read_sql_query("SELECT * FROM sales WHERE category = 'ドリンク' LIMIT 5", conn)

#### ステップ 4：グループごとに集計する（GROUP BY）

`GROUP BY` で列を指定すると、その列の値ごとにグループ化されます。`SUM()` や `COUNT()` といった集計関数と組み合わせることで、Pandas の `groupby` に相当する操作を SQL 側で実行できます。

In [ ]:
pd.read_sql_query(
    'SELECT category, SUM(quantity) AS total_qty FROM sales GROUP BY category',
    conn,
)

#### ステップ 5：接続を閉じる

DB との通信が終わったら必ず接続を閉じます。閉じ忘れるとファイルがロックされたままになる場合があります。

In [ ]:
conn.close()

---

## 2. 演習

### 2.1 背景

> これまで毎週 CSV を書き出してもらっていましたが、社内の販売 DB に直接アクセスできるようになりました。カテゴリごとの販売数合計を、販売数の多い順に並べて確認したいです。
#### リサーチクエスチョン (RQ)

SQLite データベースに接続し、SQL クエリで **カテゴリごとの販売数合計** を **降順**に並べて DataFrame として取得するには、どのような手順が必要か？

#### 仮説

ステップ 4 で学んだ `GROUP BY` に、さらに `ORDER BY total_qty DESC` を加えれば、カテゴリごとの合計値を販売数の多い順で並べた DataFrame が一度のクエリで得られるはずです。

### 2.2 分析

In [ ]:
# ============================================================
# 設計 → 実装
# ------------------------------------------------------------
# データ分析は「いきなりコードを書く」ものではありません。
# まず日本語で分析の骨組み（手順）を書き出し、
# それを一つずつ Python コードに翻訳していきます。
#
# このセルには授業中に以下の流れで記入します:
#   1. 骨組みコメントを書く（# 1. …, # 2. …, # 3. …）
#      ← 「何をするか」を日本語で明文化
#   2. 各コメントの下にコードを追記する
#      ← 日本語の手順を Python に「翻訳」
#
# 生成 AI も、骨組みが明確なほど正確なコードを返してくれます。
# この「描いてから書く」習慣を本科目で身につけましょう。
# ============================================================

# 1.

# 2.

# 3.

# 4.

# 5.


### 2.3 サマリー

以下の穴埋め項目を、演習で実行したコードの出力結果をもとに記入してください。

#### 分析結果の確認

- 販売数合計が最も多かったカテゴリは \_\_\_\_（合計 \_\_\_\_ 個）でした
- 2 位のカテゴリは \_\_\_\_（合計 \_\_\_\_ 個）でした
- 3 位のカテゴリは \_\_\_\_（合計 \_\_\_\_ 個）でした

#### 結論

上記の確認により、Python の \_\_\_\_（DB 接続）と \_\_\_\_（SQL 実行と DataFrame 化）を使うことで、データベースに格納された大量の売上データから、必要な集計結果だけを数行のコードで取り出せることを確認できました。